In [15]:
import torch
import gymnasium as gym # required for DRL
import torch.nn as nn # base class for creating neural net
import torch.optim as optim # imports PyTorch's optimization module, which provides various algorithms (like SGD, Adam) for updating neural network parameters during training.
from torch.distributions import Categorical


In [29]:
class PolicyNetwork(nn.Module):
    def __init__(self):
        super(PolicyNetwork,self).__init__() # make sure that the nn class is intialized
        self.fc = nn.Sequential( # fc is fully conected, it is just a variable name and could be names something else
                                 # Sequential creates sequential chain of layers as MDP is a sequential process
            nn.Linear(4, 128),   # Linear - takes 4 inputs and ouputs 128 dimensional features
            nn.ReLU(),           # Nonlinear activation, introducing "squashing" so the model can learn complex patterns
            nn.Linear(128,2),    # Another fully-connected layer; turns the 128-dimensional hidden layer                                       into 2 outputs (one per possible action left/right)
            nn.Softmax(dim=-1))  # Turns the 2 outputs into probabilities adding up to 1, so you can                                           sample them as actions.
    def forward(self, x):    # Defines how a state passes through (forward propagation): input x is processed by the                               self.fc block.
        return self.fc(x)

In [30]:
env = gym.make('CartPole-v1')

In [31]:
#Instantiates your policy network. Now policy is a callable object for making decisions.
policy = PolicyNetwork()

In [32]:
#Sets up the Adam optimizer to update the network's parameters during training (gradient descent), with a learning rate of 0.01.
optimizer = optim.Adam(policy.parameters(), lr=1e-2)


In [33]:
# training loop
for episode in range(1000):
    state, _ = env.reset()
    log_probs = []
    rewards = []
    done = False
    while not done:
        state_tensor = torch.from_numpy(state).float()
        probs = policy(state_tensor)
        m = Categorical(probs)
        action = m.sample()
        log_probs.append(m.log_prob(action))
        state, reward, done, _, _ = env.step(action.item())
        rewards.append(reward)
    # Compute discounted return and update policy
    discounted_rewards = []
    G = 0
    for r in reversed(rewards):
        G = r + 0.99 * G
        discounted_rewards.insert(0, G)
    discounted_rewards = torch.tensor(discounted_rewards)
    loss = -torch.stack(log_probs) * discounted_rewards
    loss = loss.sum()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
